# PhantomMap — Colab T4 quickstart

Repo: https://github.com/Evan715823/PhantomMap

**Before running**: `Runtime -> Change runtime type -> T4 GPU`. The full sweep takes ~10 h T4 time. Every script resumes from its own jsonl output, so it's safe to run across multiple sessions.

**How Claude sees your progress**: after each split finishes, download the `results/*.jsonl` file and drop it into your local `e:/wsy/vision/project/results/` folder. Claude reads that file directly and reports what it sees.

Workflow:
1. Cells 1–3: setup (once per Colab session).
2. Cell 4 (dry run): 20 samples on Qwen2.5-VL-3B to verify the pipeline before committing to the 80-min runs.
3. Cells 5–7: full runs (long).
4. Cells 8–12: local analysis (you should prefer running these on your laptop once jsonls are back).

In [ ]:
# 1. Clone the project.
!git clone https://github.com/Evan715823/PhantomMap.git /content/phantommap
%cd /content/phantommap

In [ ]:
# 2. Install dependencies. Colab already has torch + CUDA preinstalled; this only adds the VLM-specific libs.
!pip install -q -U transformers qwen-vl-utils accelerate scikit-learn seaborn scipy tqdm
!python -c "import torch; print('torch', torch.__version__, 'cuda', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '-')"

In [ ]:
# 3. Download data (POPE jsonl + AMBER query + the COCO val2014 images referenced by POPE).
#    AMBER images aren't auto-fetched; we grab them in cell 3b if needed.
!python src/download_data.py --out data

In [ ]:
# 3b. (Optional) Also fetch AMBER images, a ~300 MB zip from the AMBER repo.
#     Skip this cell if you only plan to run POPE.
!mkdir -p data/amber/images
!wget -q -c https://github.com/junyangwang0410/AMBER/raw/main/data/AMBER-image.zip -O data/amber/images.zip
!cd data/amber && unzip -q -o images.zip -d images/ && rm images.zip
!ls data/amber/images | head

In [ ]:
# 4. DRY RUN — 20 samples on Qwen2.5-VL-3B-Instruct to verify the pipeline.
#    Uses the smaller 3B model for a quick (~2 min) check that download,
#    parsing, bbox output, and logprob capture all work. If this fails,
#    we fix the bug before wasting 80 minutes on the full run.
!python src/run_vlm.py --model qwen --split pope_adversarial \
    --out results/dryrun_qwen3b.jsonl --limit 20
!echo '--- first 3 output records ---'
!head -n 3 results/dryrun_qwen3b.jsonl

In [ ]:
# 5. FULL RUN — Qwen2.5-VL-7B across all POPE splits (~80 min per split on T4).
#    Safe to Ctrl-C / disconnect; re-running resumes from the existing jsonl.
!python src/run_vlm.py --model qwen --split pope_adversarial --out results/qwen_pope_adversarial.jsonl
!python src/run_vlm.py --model qwen --split pope_popular     --out results/qwen_pope_popular.jsonl
!python src/run_vlm.py --model qwen --split pope_random      --out results/qwen_pope_random.jsonl

In [ ]:
# 6. (Optional) AMBER on Qwen2.5-VL-7B — needs cell 3b images first.
!python src/run_vlm.py --model qwen --split amber --out results/qwen_amber.jsonl

In [ ]:
# 7. FULL RUN — LLaVA-NeXT across POPE splits (~4 h total).
!python src/run_vlm.py --model llava --split pope_adversarial --out results/llava_pope_adversarial.jsonl
!python src/run_vlm.py --model llava --split pope_popular     --out results/llava_pope_popular.jsonl
!python src/run_vlm.py --model llava --split pope_random      --out results/llava_pope_random.jsonl

In [ ]:
# 8. Zip all jsonls so you can download them in one file.
#    After this cell runs, right-click results_bundle.zip in the left
#    file panel and Download, then drop the jsonls into your local
#    e:/wsy/vision/project/results/ directory.
!cd results && zip -q results_bundle.zip *.jsonl && ls -la results_bundle.zip

In [ ]:
# 9. Sanity check: aggregate POPE hallucination rates per split.
!python -c "\
import json, glob, sys; sys.path.insert(0, 'src'); from metrics import pope_stats; \
for p in sorted(glob.glob('results/*_pope_*.jsonl')): \
  recs = [json.loads(l) for l in open(p) if l.strip()]; \
  s = pope_stats(recs); \
  print(f'{p}: n={s.n} acc={s.accuracy:.3f} hallu={s.hallucination_rate:.3f} yes={s.yes_rate:.3f}')"

## Post-processing (do this LOCALLY once jsonls are back on your laptop)

Once `results/*.jsonl` are on your laptop, the rest is CPU-only and runs in seconds. Claude can drive this part directly.

```bash
# atlas
python src/atlas.py --inputs results/*.jsonl \
  --out-fig report/figures/fig3_atlas.pdf --out-stats results/atlas_stats.json

# detector per model
python src/detector.py --inputs results/qwen_*.jsonl \
  --model-filter "Qwen/Qwen2.5-VL-7B-Instruct" \
  --out results/detector_qwen.json
python src/detector.py --inputs results/llava_*.jsonl \
  --model-filter "llava-hf/llava-v1.6-mistral-7b-hf" \
  --out results/detector_llava.json

# figures
python src/make_fig2_method.py --out report/figures/fig2_method.pdf
python src/make_figures.py --predictions results/*.jsonl \
  --detector-metrics results/detector_qwen.json results/detector_llava.json

# build PDF
cd report && pdflatex report.tex && bibtex report && pdflatex report.tex && pdflatex report.tex
```